# 07 - Top configurations

Reads the metrics of scripts 06 and 07 and, for each k, shows the best configurations of one study next to its baselines. Configurations that switch regimes more than twice as often as the baseline are discarded, the rest are ranked by the F1 score of expensive days on validation, and the same configurations are shown on test.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd

from src.metrics import comparison_table, persistence_factor, ranking

## 0. Configuration

`study` is the study to show. For the calendar week studies use `metrics_file = "metrics/calweek/regime_metrics.csv"` and `reference = "level"`.
- `top` is the number of configurations shown per k.

In [ ]:
study = "daily"
top = 3
k_values = [3, 4, 5]
choose_period, report_period = "val", "test"
metrics_file, reference = "metrics/regime_metrics.csv", "level 7d"

## 1. Top configurations

In [ ]:
rows = pd.read_csv(Path("..") / metrics_file)
unit = "weeks" if reference == "level" else "days"  # each row of the calendar-week metrics is a week
if study not in set(rows["study"]):
    raise ValueError(f"{study} is not in {metrics_file}: {sorted(set(rows['study']))}")

baseline_labels = {"features": "Baseline", "view": "View"}
model_labels = {"rank_c": "Rank", "z_dim": "z_dim", "lam": "lam", "beta": "beta", "features": "Features",
                "view": "View", "seeds": "Seeds"}
keys = ["model", "study", "z_dim", "lam", "beta", "features", "view"]

for k in k_values:
    stats = ranking(rows, k, choose_period, reference)
    limit = stats["switches_limit"].iloc[0]
    candidates = stats[(stats["study"] == study) & (stats["model"] != "level_only")]
    passed = candidates.dropna(subset=["rank_c"]).sort_values("rank_c").head(top)
    passed = passed.assign(rank_c=range(1, len(passed) + 1))
    print(f"k = {k}: {study}, {len(candidates)} configurations, {len(candidates.dropna(subset=['rank_c']))} switch at most "
          f"{persistence_factor:g} x the {reference} baseline in {choose_period} ({limit:.1f} per year)")
    for period in (choose_period, report_period):
        period_stats = stats if period == choose_period else ranking(rows, k, period, reference)
        print(f"Period: {period}" + (" (chooses the models)" if period == choose_period else " (reported)"))
        print(comparison_table(period_stats[period_stats["model"] == "level_only"], baseline_labels, unit))
        if passed.empty:
            print(f"No configuration of {study} experiments passed the switches filter with k = {k}.")
        else:
            shown = passed[keys + ["rank_c"]].merge(period_stats.drop(columns=["rank_c"]), on=keys)
            print(comparison_table(shown.sort_values("rank_c"), model_labels, unit))
    print()

In [ ]:
features_shown = None  # None shows every input; "latents + level" shows only the unsmoothed one
for k in k_values:
    for period in (choose_period, report_period):
        stats = ranking(rows, k, period, reference)
        candidates = stats[(stats["study"] == study) & (stats["model"] != "level_only")]
        if features_shown:
            candidates = candidates[candidates["features"] == features_shown]
        print(f"k = {k}, {period}: {study}, all configurations (filter limit {stats['switches_limit'].iloc[0]:.1f} per year)")
        print(comparison_table(candidates.sort_values("expensive_f1_pct_mean", ascending=False), model_labels, unit))